In [ ]:
import h5py
import numpy as np
import kikuchipy as kp  # modified to enable dealing with large up2 files
import matplotlib.pyplot as plt
from jupyterlab_h5web import H5Web

# path = "3D_EBSD_CathodeParticle_RawPatterns_Slice 1/3D_EBSD_CathodeParticle_RawPatterns_Slice (1).up2"
# path = "Agate_Quartz_AGH.h5oina"
# path = "718_microVol_053.up2"
path = "/mnt/production/nexus_paper/300/Patterns.up2"
output = "/mnt/production/nexus_paper/ElectronDiffraction.nxs"

In [ ]:
dat = kp.load(path, lazy=True)  # note that for up2 edax_binary is not lazy though

In [ ]:
dat
# print(dat.axes_manager)
# print(f"{type(dat.data)}, {dat.data.dtype}")

One time generation, compositing other metadata later.

In [ ]:
n_chunks: int = 176800
n_selection: int = 1768  # first that many pattern
ebsd_camera_resolution: tuple[int, int] = (446, 446)
with h5py.File(output, "w") as h5w:
    # https://zenodo.org/records/11403783
    # optimal overwriting the auto chunker for this dataset
    #
    prefix: str = "/entry1/measurement/event1/image1/stack_2d"
    grp = h5w.create_group(prefix)
    grp.attrs["NX_class"] = "NXdata"
    grp.attrs["signal"] = "intensity"
    grp.attrs["axes"] = ["indices_image", "axis_j", "axis_i"]
    grp.attrs["axis_i_indices"] = np.uint32(0)
    grp.attrs["axis_j_indices"] = np.uint32(1)
    grp.attrs["indices_image_indices"] = np.uint32(2)
    dst = h5w.create_dataset(
        f"{prefix}/title",
        data="Kikuchi pattern stack",
    )
    dst = h5w.create_dataset(
        f"{prefix}/intensity",
        compression="gzip",
        compression_opts=9,
        chunks=(int((1 * 1024 ** 2) / (2 * 446 * 446)), 446, 446),
        data=dat.inav[0:1000:],  # n_selection],
    )

In [ ]:
del dat
# H5Web(output)

Axes annotations

In [ ]:
with h5py.File(output, "a") as h5w:
    prefix: str = "/entry1/measurement/event1/image1/stack_2d"
    for dim_idx, dim_suffix in enumerate(["i", "j"]):
        dst = h5w.create_dataset(
            f"{prefix}/axis_{dim_suffix}",
            compression="gzip",
            compression_opts=9,
            chunks=(ebsd_camera_resolution[dim_idx],),
            data=np.asarray(np.linspace(0, ebsd_camera_resolution[dim_idx] - 1, num=ebsd_camera_resolution[dim_idx], endpoint=True), np.uint32),
        )
    dst = h5w.create_dataset(
        f"{prefix}/indices_image",
        compression="gzip",
        compression_opts=9,
        chunks=(n_selection,),
        data=np.asarray(np.linspace(0, n_selection - 1, num=n_selection, endpoint=True), np.uint32),  # TODO sub-sampling not considered
    )

In [ ]:
# del dat
H5Web(output)

Class annotations

In [ ]:
with h5py.File(output, "a") as h5w:
    prefix: str = "/entry1/measurement/event1/image1/stack_2d"
    h5w["/entry1"].attrs["NX_class"] = "NXentry"
    h5w["/entry1/measurement"].attrs["NX_class"] = "NXem_measurement"
    h5w["/entry1/measurement/event1"].attrs["NX_class"] = "NXem_event_data"
    h5w["/entry1/measurement/event1/image1"].attrs["NX_class"] = "NXimage"
    dst = h5w.create_dataset("/entry1/definition", data="NXem")
    dst.attrs["version"] = f"{get_nexus_version()}"

In [ ]:
# H5Web(output)

Path to the default plot

In [ ]:
with h5py.File(output, "a") as h5w:
    prefix: str = "/entry1/measurement/event1/image1/stack_2d"
    h5w["/"].attrs["default"] = "entry1"
    h5w["/entry1"].attrs["NX_class"] = "NXentry"
    h5w["/entry1"].attrs["default"] = "measurement"
    h5w["/entry1/measurement"].attrs["NX_class"] = "NXem_measurement"
    h5w["/entry1/measurement"].attrs["default"] = "event1"
    h5w["/entry1/measurement/event1"].attrs["NX_class"] = "NXem_event_data"
    h5w["/entry1/measurement/event1"].attrs["NX_class"] = "NXem_event_data"
    h5w["/entry1/measurement/event1"].attrs["default"] = "image1"
    h5w["/entry1/measurement/event1/image1"].attrs["NX_class"] = "NXimage"
    h5w["/entry1/measurement/event1/image1"].attrs["default"] = "stack_2d"

***

Manual metadata

In [ ]:
with h5py.File(output, "a") as h5w:
    # detailed in https://zenodo.org/records/11403783
    # TODO EBSD 300.3350ea00e300a9c6168947a729b632a7e505b5aba53041e244638514f7a9dfec.{ang|mtex.h5}
    """
    grp = h5w.create_group("/entry1/measurement/event1/instrument")
    grp.attrs["NX_class"] = "NXem_instrument"
    grp = h5w.create_group("/entry1/measurement/event1/instrument/ebeam_column")
    grp.attrs["NX_class"] = "NXcomponent"
    grp = h5w.create_group("/entry1/measurement/event1/instrument/ebeam_column/electron_source")
    grp.attrs["NX_class"] = "NXsource"
    dst = h5w.create_dataset("/entry1/measurement/event1/instrument/ebeam_column/electron_source/voltage", data=np.float64(20.))
    dst.attrs["units"] = "kilovolt"
    grp = h5w.create_group("/entry1/measurement/event1/instrument/optics")
    grp.attrs["NX_class"] = "NXem_optical_system"
    dst = h5w.create_dataset("/entry1/measurement/event1/instrument/optics/working_distance", data=np.float64(20.))
    dst.attrs["units"] = "millimeter"
    dst = h5w.create_dataset("/entry1/measurement/event1/instrument/optics/magnification", data=np.float64(200.))
    """
    grp = h5w.create_group("/entry1/measurement/instrument")
    grp.attrs["NX_class"] = "NXem_instrument"
    grp = h5w.create_group("/entry1/measurement/instrument/fabrication")
    grp.attrs["NX_class"] = "NXfabrication"
    dst = h5w.create_dataset("/entry1/measurement/instrument/fabrication/vendor", data="JEOL")
    dst = h5w.create_dataset("/entry1/measurement/instrument/fabrication/model", data="JEOL JSM-IT300HR")

In [ ]:
# H5Web(output)

ROI TODO align with SHT

In [ ]:
with h5py.File("/mnt/production/nexus_paper/300.3350ea00e300a9c6168947a729b632a7e505b5aba53041e244638514f7a9dfec.ang.mtex.h5", "r") as h5r:
    with h5py.File(output, "a") as h5w:
        # pynxtools-microstructure
        grp = h5w.create_group("/entry1/roi1")
        grp.attrs["NX_class"] = "NXroi_process"
        grp = h5w.create_group("/entry1/roi1/ebsd")
        grp.attrs["NX_class"] = "NXem_ebsd"
        grp = h5w.create_group("/entry1/roi1/ebsd/indexing")
        grp.attrs["NX_class"] = "NXprocess"

user name, citation, etc. annotations

In [ ]:
with h5py.File(output, "a") as h5w:
    # pynxtools-microstructure
    grp = h5w.create_group("/entry1/user1")
    grp.attrs["NX_class"] = "NXuser"
    dst = h5w.create_dataset("/entry1/user1/name", data="Markus Kühbach")
    grp = h5w.create_group("/entry1/cite1")
    grp.attrs["NX_class"] = "NXcite"
    dst = h5w.create_dataset("/entry1/cite1/author", data="Bennett IV, Thomas J. and Taleff, Eric M.")
    dst = h5w.create_dataset("/entry1/cite1/doi", data="10.5281/zenodo.11403783")
    dst = h5w.create_dataset("/entry1/cite1/title", data="Electron Backscatter Diffraction Patterns from Titanium-added Interstitial-free Steel Containing Subgrains")
    dst = h5w.create_dataset("/entry1/experiment_description", data="Electron Backscatter Diffraction Patterns from Titanium-added Interstitial-free Steel Containing Subgrains,\nBennett IV, Thomas J. and Taleff, Eric M.,\n10.5281/zenodo.11403783")

    # https://link.springer.com/article/10.1007/s11661-023-07256-w/tables/1
    grp = h5w.create_group("/entry1/sample")
    grp.attrs["NX_class"] = "NXsample"
    dst = h5w.create_dataset("/entry1/sample/atom_types", data="Mn, Al, Ti, Cr, Cu, Ni, S, P, Nb, Si, Mo, As, N, C, Pb, Sn, V, Sb, B, Ca, Fe")

In [ ]:
H5Web(output)
# H5Web("/mnt/production/nexus_paper/300.3350ea00e300a9c6168947a729b632a7e505b5aba53041e244638514f7a9dfec.ang.mtex.h5")